# !!!主要解释!!!必读！！！：
1. 本文件是从09 neural_network_and_deep_learning 复制来，修改后，为的是：用新的tensorflow 2.13版本重新训练模型（原版本quant环境下的是2.21版本）
2. 为了达到目的，给当前文件所在的目录创建了新的uv虚拟环境，并安装了tensorflow 2.13.1
还要安装这些：    uv pip install scipy    uv pip install tensorflow-serving-api==2.13.1     uv pip install  grpcio==1.74.0
uv pip install keras-image-helper 
4. 给uv安装了 pip install ipykernel，并配置了kernel：python -m ipykernel install --user --name mlops-tf13 --display-name "Python (mlops TF2.13)"
5. 让当前ipynb文件选择‘mlops-tf13’这个kernel，进行模型的训练。
6. 注意：第3个cell不要执行，不要查看版本号，切记。

##  GPU已经可以使用，重要提示：除了使用kimi3来帮助解决uv环境中，使用gpu训练模型的配置方法外，
- 还有就是要在uv安装：uv pip install jupyter jupyterlab ipykernel
-在这个 venv 里启动 Jupyter。这样 kernel 一启动就带着完整的 LD_LIBRARY_PATH，TF 就能找到所有 CUDA/cuDNN 库。
## 所以11_k8s的目录需要挪到 neural_network_deep_learning  这个目录中

In [ ]:
## 这就是 neural_network_deep_learning  虚拟机，完好状态下，依赖库的情况
(neural_network_deep_learning) (quant) root@MS-CEVXSRKPSHOI:/mnt/d/develop/mlops/ml/11_k8s# uv pip list | grep -E "tensorflow|tensorflow-serving-api|scipy|grpcio|keras-image-helper|numpy"
Using Python 3.11.15 environment at: /mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv
grpcio                       1.74.0
keras-image-helper           0.0.2
numpy                        1.24.3
scipy                        1.13.1
tensorflow                   2.13.1
tensorflow-estimator         2.13.0
tensorflow-io-gcs-filesystem 0.37.1
tensorflow-serving-api       2.13.1

In [ ]:
git clone https://github.com/alexeygrigorev/clothing-dataset-small.git 
按照课程视频的要求，clone了这个仓库，放到和当前学习内容目录，平行的目录中，防止git混乱
当前学习记录的目录：/mnt/d/develop/mlops/ml
新的目录：/mnt/d/develop/mlops/clothing-dataset-small

In [1]:
import os
import sys

print("=" * 50)
print("Python:", sys.executable)
print("LD_LIBRARY_PATH:", os.environ.get('LD_LIBRARY_PATH', 'NOT SET'))
print("=" * 50)

# 检查 sitecustomize 是否被加载
try:
    import sitecustomize
    print("sitecustomize loaded from:", sitecustomize.__file__)
except ImportError:
    print("sitecustomize NOT loaded")

# 检查 libcuda.so 是否能直接加载
import ctypes
try:
    ctypes.CDLL('/usr/lib/wsl/lib/libcuda.so')
    print("libcuda.so: OK")
except Exception as e:
    print("libcuda.so FAILED:", e)

# 检查 libcudart.so.11.0
try:
    ctypes.CDLL('/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/nvidia/cuda_runtime/lib/libcudart.so.11.0')
    print("libcudart.so.11.0: OK")
except Exception as e:
    print("libcudart.so.11.0 FAILED:", e)

Python: /mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/bin/python
LD_LIBRARY_PATH: /mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/nvidia/cudnn/lib:/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/nvidia/cublas/lib:/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/nvidia/cufft/lib:/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/nvidia/curand/lib:/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/nvidia/cusolver/lib:/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/nvidia/cusparse/lib:/usr/lib/wsl/lib
sitecustomize loaded from: /mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/sitecustomize.py
libcud

In [2]:
import sys
print(sys.executable)

/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/bin/python


In [3]:
!python -c "import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))"

2026-09-16 09:42:43.360550: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-16 09:42:51.879571: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-09-16 09:43:16.148762: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-16 09:43:16.358455: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-16 09:43:16.358563: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_exec

In [4]:
# ===== WSL2 + TF 2.13.1 GPU 强制修复（必须放在文件最开头，import tensorflow 之前）=====
VENV_PATH = '/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/nvidia'

# 1. 强制设置库路径
os.environ['LD_LIBRARY_PATH'] = ':'.join([
    f'{VENV_PATH}/cuda_runtime/lib',
    f'{VENV_PATH}/cudnn/lib',
    f'{VENV_PATH}/cublas/lib',
    f'{VENV_PATH}/cufft/lib',
    f'{VENV_PATH}/curand/lib',
    f'{VENV_PATH}/cusolver/lib',
    f'{VENV_PATH}/cusparse/lib',
    '/usr/lib/wsl/lib',
])


In [5]:
# 2. 用 ctypes 预加载关键 CUDA/cuDNN 库（绕过 TensorFlow 的延迟加载机制）
import ctypes
ctypes.CDLL('/usr/lib/wsl/lib/libcuda.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(f'{VENV_PATH}/cuda_runtime/lib/libcudart.so.11.0', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(f'{VENV_PATH}/cudnn/lib/libcudnn.so.8', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(f'{VENV_PATH}/cublas/lib/libcublas.so.11', mode=ctypes.RTLD_GLOBAL)

<CDLL '/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/nvidia/cublas/lib/libcublas.so.11', handle 590a44d6b090 at 0x73e69f478c10>

In [6]:
# 3. 限制显存增长（2GB 显存保命设置）
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f"✅ GPU 已启用: {gpus[0]}")
else:
    print("❌ GPU 未找到，将使用 CPU")

2026-09-16 09:43:39.135441: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-16 09:43:48.421996: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


✅ GPU 已启用: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


2026-09-16 09:44:15.108242: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-16 09:44:15.208013: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-16 09:44:15.208120: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.


In [7]:
# 4. 验证 cuDNN 是否加载
print(f"TF version: {tf.__version__}")
print(f"CUDA built with: {tf.sysconfig.get_build_info().get('cuda_version', 'unknown')}")
print(f"cuDNN built with: {tf.sysconfig.get_build_info().get('cudnn_version', 'unknown')}")

TF version: 2.13.1
CUDA built with: 11.8
cuDNN built with: 8


In [8]:
import sys
print(sys.executable)          # 看解释器路径
import tensorflow as tf
print(tf.__version__)          # 看 TF 版本
import keras
print(keras.__version__) # 应输出 2.13.x
import numpy as np
import scipy
from scipy import ndimage
print(np.__version__, scipy.__version__)
print(ndimage)

/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/bin/python
2.13.1
2.13.1
1.24.3 1.13.1
<module 'scipy.ndimage' from '/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/scipy/ndimage/__init__.py'>


In [9]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [10]:
import keras
print(keras.__version__)
# Keras 是一个高级神经网络 API（应用程序编程接口），它提供了：
# 更简洁、易用的语法来构建深度学习模型
# 模块化的组件（层、优化器、损失函数等）
# 快速原型设计能力

2.13.1


In [11]:
from tensorflow.keras.applications.xception import Xception   #使用 keras中的Xception 模型
from tensorflow.keras.applications.xception import preprocess_input  # 数据预处理
from tensorflow.keras.applications.xception import decode_predictions # 解码预测结果

In [12]:
# 模型会下载到：~/.keras/models/

In [13]:
# ，用于在训练过程中动态生成批量图像数据，并对图像进行随机变换（如旋转、缩放、翻转等），以扩充数据集、提升模型的泛化能力。
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [14]:
# 在模型训练过程中设置一个 自动保存检查点（Checkpoint） 的回调函数
chechpoint = keras.callbacks.ModelCheckpoint(
    'xception_v1_{epoch:02d}_{val_accuracy:.3f}.h5',  #一个动态生成的文件名
    save_best_only=True, #只保存性能最好的那一个（而不是每轮都存）。
    monitor='val_accuracy',  #监控指标：验证集准确率（val_accuracy）。
    mode='max' # 优化方向：'max' 表示数值越大越好（准确率当然越高越好）
)

# 8.11 Training a larger model
Train a 299x299 model

In [15]:
def make_model811(input_size=150, learning_rate=0.01, size_inner=100,
               droprate=0.5):

    base_model = Xception(
        weights='imagenet',
        include_top=False,
        input_shape=(input_size, input_size, 3)
    )

    base_model.trainable = False

    #########################################

    inputs = keras.Input(shape=(input_size, input_size, 3))
    base = base_model(inputs, training=False)
    vectors = keras.layers.GlobalAveragePooling2D()(base)
    
    inner = keras.layers.Dense(size_inner, activation='relu')(vectors)
    drop = keras.layers.Dropout(droprate)(inner)
    
    outputs = keras.layers.Dense(10)(drop)
    
    model = keras.Model(inputs, outputs)
    
    #########################################

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    loss = keras.losses.CategoricalCrossentropy(from_logits=True)

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['accuracy']
    )
    
    return model

In [16]:
input_size = 299

In [17]:
# 关闭 XLA（减少 CPU 开销）
#import os
#os.environ['TF_ENABLE_XLA'] = '0'

# 混合精度训练（FP16）  MX570 支持 FP16，可以让计算更快，从而提升利用率
#from tensorflow.keras.mixed_precision import set_global_policy
#set_global_policy('mixed_float16')

# 配置增强参数
train_gen4 = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    shear_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

train_ds4 = train_gen4.flow_from_directory(
    '../../clothing-dataset-small/train',
    target_size=(input_size, input_size),
    batch_size=4
)

# 用 from_generator 但限制 prefetch 为 1，提高GPU使用率
#train_ds4 = tf.data.Dataset.from_generator(
#    lambda: train_ds4_raw,
#    output_signature=(
#        tf.TensorSpec(shape=(None, 299, 299, 3), dtype=tf.float32),
#        tf.TensorSpec(shape=(None, 10), dtype=tf.float32)
#    )
#).prefetch(1)  # 用 1 而不是 AUTOTUNE


val_gen4 = ImageDataGenerator(preprocessing_function=preprocess_input)

val_ds4 = val_gen4.flow_from_directory(
    '../../clothing-dataset-small/validation',
    target_size=(input_size, input_size),
    batch_size=4,
    shuffle=False
)

Found 3068 images belonging to 10 classes.
Found 341 images belonging to 10 classes.


In [18]:
checkpoint = keras.callbacks.ModelCheckpoint(
    'xception_v4_1_{epoch:02d}_{val_accuracy:.3f}.h5',
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [ ]:
learning_rate = 0.0005
size = 100
droprate = 0.2


model = make_model811(
    input_size=input_size,
    learning_rate=learning_rate,
    size_inner=size,
    droprate=droprate
)

history = model.fit(train_ds4, epochs=50, validation_data=val_ds4,
                   callbacks=[checkpoint])

2026-09-16 09:45:04.432950: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-16 09:45:04.433117: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-16 09:45:04.433163: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-16 09:45:06.003159: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-16 09:45:06.003358: I tensorflow/compile

Epoch 1/50


2026-09-16 09:45:14.379986: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600
2026-09-16 09:45:16.705987: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-09-16 09:45:17.966645: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x73e4dc16f860 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-09-16 09:45:17.966812: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce MX570, Compute Capability 8.6
2026-09-16 09:45:18.382427: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:255] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-09-16 09:45:19.388701: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the proc

767/767 [==============================] - ETA: 0s - loss: 0.7248 - accuracy: 0.7595

/mnt/d/develop/mlops/ml/neural_network_deep_learning/.venv/lib/python3.11/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


767/767 [==============================] - 248s 310ms/step - loss: 0.7248 - accuracy: 0.7595 - val_loss: 0.4632 - val_accuracy: 0.8475
Epoch 2/50
767/767 [==============================] - 239s 312ms/step - loss: 0.4256 - accuracy: 0.8458 - val_loss: 0.4395 - val_accuracy: 0.8622
Epoch 3/50
  1/767 [..............................] - ETA: 12:23 - loss: 0.0503 - accuracy: 1.0000

In [ ]:
hist = history.history
plt.plot(hist['val_accuracy'], label='val')
plt.plot(hist['accuracy'], label='train')

plt.legend()

In [12]:
# 我们保留 最优秀的模型，删除掉差的模型，本节课程结束了，欢迎大家来到神经网络的世界
- 我得出了最佳模型，在第45次迭代的时候：val_accuracy: 0.8915

SyntaxError: invalid character '，' (U+FF0C) (2363563934.py, line 2)

# 8.12 Using the model
- Loading the model
- Evaluating the model
- Getting predictions

In [2]:
import tensorflow as tf
from tensorflow import keras

In [5]:
model_last = keras.models.load_model('xception_v4_1_45_0.891.h5')

I0000 00:00:1788528331.577381   72083 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1351 MB memory:  -> device: 0, name: NVIDIA GeForce MX570, pci bus id: 0000:02:00.0, compute capability: 8.6


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator 
from tensorflow.keras.applications.xception import preprocess_input  

In [9]:
test_gen_last = ImageDataGenerator( preprocessing_function=preprocess_input)  

test_ds = test_gen_last.flow_from_directory(
    '../clothing-dataset-small/test',
    target_size = (299,299),
    batch_size = 4,
    shuffle=False
)

Found 372 images belonging to 10 classes.


In [11]:
model_last.evaluate(test_ds)

I0000 00:00:1788528563.715675   72083 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1788528567.413505   78692 service.cc:153] XLA service 0x71d5480350f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788528567.413553   78692 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce MX570, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.6.0; Toolkit: 12.5.0; DNN: 9.25.1)
I0000 00:00:1788528567.524838   78692 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1788528568.495224   78692 cuda_dnn.cc:461] Loaded cuDNN version 92501
I0000 00:00:1788528591.925139   78692 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


93/93 ━━━━━━━━━━━━━━━━━━━━ 42s 144ms/step - accuracy: 0.8871 - loss: 0.4086


[0.4085538983345032, 0.8870967626571655]

### 总结一下下
93步，完成测试；372 / 93 = 4 (batch_size)
- loss: 0.4086   损失值，越低越好
- accuracy: 0.8871   准确度，88.71%
- 模型泛化能力很好，验证集最佳准确率：89.1%（训练时）
- 测试集准确率：88.71%
- 两者非常接近（只差 0.4%），说明模型没有过拟合，在未见过的数据上表现稳定。

In [ ]:
# 测试单个图片 
path = '../clothing-dataset-small/test/pants/15a1e02b-ad2a-4b9f-9d44-0e4d260594a8.jpg' 

In [ ]:
from tensorflow.keras.preprocessing.image import load_img 

In [20]:
img_last = load_img(path,target_size = (299,299))

In [23]:
import numpy as np

In [24]:
x_last = np.array(img_last)

In [28]:
X_last = np.array([x_last])

In [29]:
X_last.shape

(1, 299, 299, 3)

In [30]:
X_last = preprocess_input(X_last)

In [31]:
pred = model_last.predict(X_last)

1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step


In [35]:
pred[0]

array([ -0.47377864, -14.1071415 ,  -6.6320066 ,  -8.9244585 ,
        12.71025   ,  -3.0698037 ,  -8.024976  ,   1.5428455 ,
       -11.129196  ,  -6.82832   ], dtype=float32)

In [33]:
classes = ['dress',
 'hat',
 'longsleeve',
 'outwear',
 'pants',
 'shirt',
 'shoes',
 'shorts',
 'skirt',
 't-shirt']

In [36]:
dict(zip(classes,pred[0]))

{'dress': np.float32(-0.47377864),
 'hat': np.float32(-14.1071415),
 'longsleeve': np.float32(-6.6320066),
 'outwear': np.float32(-8.9244585),
 'pants': np.float32(12.71025),
 'shirt': np.float32(-3.0698037),
 'shoes': np.float32(-8.024976),
 'shorts': np.float32(1.5428455),
 'skirt': np.float32(-11.129196),
 't-shirt': np.float32(-6.82832)}

In [ ]:
# 结果：'pants': np.float32(12.71025),  可以理解12.7是概率，这个值真的很高，远远高于其他值

# 8.13 Summary
- We can use pre-trained models for general image classification
- Convolutional layers let us turn an image into a vector
- Dense layers use the vector to make the predictions
- Instead of training a model from scratch, we can use transfer learning and re-use already trained convolutional lavers

# 8.14 Explore more
## 这是我认为alexey说的比较重要的事情，我们一定是要学习的
## Use ***PyTorch*** or MXNet instead of TensorFlow/Keras
## 其他可以使用的模型
如果你想追求更高的精度：可以尝试 ResNet 或 DenseNet。但要特别注意，它们的参数量可能很大，需搭配更小的 batch_size，这会拉长训练时间。

如果你想提升训练速度：MobileNet 系列是首选，它足够轻量，能在你的 GPU 上用更大的 batch_size，从而加快训练。之前的轻量化研究也常用 MobileNetV2 来替代 Xception。

如果你想兼顾不错的精度和效率：可以探索 EfficientNet 或 ShuffleNet。尤其是 ShuffleNetV2，在不少测试中都展现出很好的精度-效率平衡。